# multimodal

> NeoMME-260M under the sysone head (its tokens, image rows built with two-axis positions, and its methods of sysone's generic functions), and the multimodal application, which takes its defaults from whichever encoder it runs.

In [ ]:
#| default_exp multimodal

In [ ]:
#| export
import numpy as np
import torch

from sysone.core import *
from sysone.template import *
from sysone.data import TypedDecisions
from sysone.models import *
from sysone.learner import PRESETS, Learner, default_preset
from sysone.learner import decision_learner as _decision_learner
from transformers import NeoMMEConfig, NeoMMEProcessor
from sysone.inference import Decider
from sysone.cache import FeatureCache        # and the verbs the cache adds to Learner, `fit_linear` among them

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail, test_ne
import json, logging, subprocess, sys, tempfile
from pathlib import Path
from PIL import Image as PILImage
logging.getLogger("transformers").setLevel(logging.ERROR)
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()

This notebook is the record of how an encoder was added, and the outline to copy for the next one: load and inspect it, build a row by hand, write the template preset, fill in the spec, run one forward pass, train a head, and write down what did not work.

## 1. Load and inspect

[NeoMME-260M](https://huggingface.co/Hcompany/NeoMME-260M) (Apache-2.0, in transformers since 5.17) is a bidirectional encoder over text and page images: 17 layers, width 1,024, a 16,384-position context, trained by masked diffusion, so `<mask>` is the token it was taught to fill. Its tokenizer has no cls or sep token and adds nothing to the text; a document starts with `<doc>`, and an image is `<img>` followed by a patch grid, one `<img>` per 32 × 32 patch and a `<row>` after each row of patches, with two-axis positions. The fixture is a random 2-layer NeoMME with a small tokenizer that keeps NeoMME's special tokens at NeoMME's ids, and the real image processor:

In [ ]:
FIX = "fixtures/tiny-neomme"
spec = EncoderSpec.from_pretrained(FIX)
tok, proc = spec.tokenizer, spec.processor
spec, {t: getattr(tok, f"{t}_token_id") for t in ("pad", "mask", "document", "image", "row")}, tok.cls_token, tok.sep_token

(EncoderSpec(fixtures/tiny-neomme: neomme, width 64, context 16384, multimodal, template neomme),
 {'pad': 0, 'mask': 4, 'document': 5, 'image': 6, 'row': 8},
 None,
 None)

In [ ]:
test_eq((spec.family, spec.modality, spec.template.name), ("neomme", "multimodal", "neomme"))
test_eq(tok("a mask here", add_special_tokens=True)["input_ids"], tok("a mask here", add_special_tokens=False)["input_ids"])

The processor lays an image out as a document of its own, `<doc><img>` then the grid, with positions (row, column): the marker at (1, 1), patch (r, c) at (2 + r, 2 + c). It rejects a row that mixes text and an image, which is exactly what a decision row is, so sysone builds those rows itself and uses the processor's image half, the patches and the grid size:

In [ ]:
shot = PILImage.open("fixtures/shot.png")
ref = proc(images=[shot], return_tensors="pt")
ref["input_ids"][0].tolist(), ref["position_ids"][:, 0].tolist(), tuple(ref["pixel_values"].shape)

([5, 6, 6, 6, 8, 6, 6, 8],
 [[0, 1, 2, 2, 2, 3, 3, 3], [0, 1, 2, 3, 4, 2, 3, 4]],
 (4, 3072))

::: {.callout-note title="Finding: NeoMME's processor cannot build a decision row"}
A decision row is text and an image in one sequence: the question and its anchored options, and the page. NeoMME's processor takes a text or an image, one per row, and refuses anything else, whichever way round; so sysone lays the row out itself and asks the processor only for the image half: the patches and the grid size.
:::

In [ ]:
test_fail(lambda: proc(text=["<doc>Is the save button visible?"], images=[shot]), contains="image prompts")
test_fail(lambda: proc(text=["<doc><img>Is it visible?"], images=[shot]), contains="token layout")

## 2. A row by hand

A NeoMME decision row with the page first: the page's grid, then its text, then the question and its options, each `<mask>` directly followed by its option's text (NeoMME's tokenizer merges a space into the word after it, and its masks replaced whole tokens in pretraining), sections separated by newlines. Text after the grid returns to diagonal positions, starting past both grid axes, as in NeoMME's image-text pairs. The positions of the whole row:

In [ ]:
#| export
def neomme_positions(n:int, image_at:int|None=None, h:int=0, w:int=0) -> torch.Tensor:
    "NeoMME's two-axis position ids for a row of `n` tokens whose image marker sits at `image_at` (an h × w grid)"
    if image_at is None: return torch.arange(n).expand(2, -1).clone()
    d = image_at - 1                                            # the <doc> that opens the image
    pos = torch.zeros(2, n, dtype=torch.long)
    pos[:, :image_at + 1] = torch.arange(image_at + 1)          # text before, <doc> and the marker: the diagonal
    r = torch.arange(h).repeat_interleave(w + 1); c = torch.arange(w + 1).repeat(h)
    g0, g1 = image_at + 1, image_at + 1 + h * (w + 1)
    pos[0, g0:g1], pos[1, g0:g1] = d + 2 + r, d + 2 + c         # patches, and each row's <row> after them
    s = d + 2 + max(h, w + 1)                                   # text after the grid starts past both axes
    pos[:, g1:] = torch.arange(s, s + n - g1)
    return pos

What those positions look like for a page whose grid is 2 × 3 patches, followed by text:

![NeoMME's two-axis positions for an image block followed by text](images/neomme_positions.svg)

*Drawn with `neomme_positions` in [figures](90_figures.ipynb#neommes-two-axis-positions).*

In [ ]:
p = neomme_positions(len(ref["input_ids"][0]) + 3, 1, 2, 2)
test_eq(p[:, :8], ref["position_ids"][:, 0])
p

tensor([[0, 1, 2, 2, 2, 3, 3, 3, 5, 6, 7],
        [0, 1, 2, 3, 4, 2, 3, 4, 5, 6, 7]])

## 3. The template

The `neomme` presets are in `sysone.template`. `build_processor_rows` is NeoMME's method of `processor_rows`, the generic function a processor-backed builder calls: the row builder's own budgets and layout for the text, with the image block, `<doc>`, `<img>` and the grid, placed at the template's `{image}` field and counted against `max_len` (never truncated; an image too large for the row is an error). Every row carries `position_ids` and `pixel_values` (empty for a text-only row), so text and image rows mix in one batch.

In [ ]:
#| export
def image_block(builder:RowBuilder, image):
    "The ids, patches and grid of one image: `<doc>`, `<img>`, then a row of patch placeholders and `<row>` per grid row"
    tok = builder.tok
    feat = builder.processor.image_processor(images=[image], return_tensors="pt")
    h, w = (int(v) for v in feat["image_grid_hw"][0])
    ids = [tok.document_token_id, tok.image_token_id] + ([tok.image_token_id] * w + [tok.row_token_id]) * h
    return ids, feat["pixel_values"], h, w

def build_processor_rows(processor:NeoMMEProcessor, builder:RowBuilder, record:dict, *, skip_errors:bool=False) -> list:
    "Rows of a transformed record for a NeoMME-style processor: text by the builder's budgets, the image as a patch grid"
    images = record.get("images") or []
    if len(images) > 1: raise ValueError("a NeoMME row holds one image")
    tok, ps = builder.tok, processor.image_processor.patch_size
    for name in ("document_token", "row_token"):
        s = getattr(tok, name, None)
        if s and s not in builder.extra_scrub: builder.extra_scrub.append(s)
    if images: ids_img, pixels, h, w = image_block(builder, images[0])
    else: ids_img, pixels, h, w = None, torch.zeros(0, 3 * ps * ps), 0, 0
    gold, rows = record.get("gold") or {}, []
    for qid, q, ids, markers, image_at, spans in builder._layout(record, ids_img, skip_errors):
        target, label = q.target(gold.get(qid))
        rows.append(Row(ids, markers, q.qtype, target, label, meta={"qid": qid, "case": record.get("id")},
                        extras={"position_ids": neomme_positions(len(ids), image_at, h, w), "pixel_values": pixels},
                        spans=spans, answerable=q.answerable(gold.get(qid))))
    return rows

processor_rows.register(build_processor_rows)

In [ ]:
qs = {"target": {"type": "choice", "instructions": "Which element is clicked next?",
                 "criteria": {"1": "[1] Add to cart (button)", "2": "[2] Size M (radio)", "3": "[3] Help (link)"}}}
page_first = spec.builder(RowTemplate.preset("neomme_page_first", max_len=256), tfms=[Image(side=64)])
rec = {"state": {"image": "fixtures/shot.png", "text": "goal: buy the blue shirt in M"}, "questions": qs, "gold": {"target": {"label": "2"}}}
r = page_first.build_record(rec)[0]
print(show_row(r, tok)); r.extras["position_ids"][:, :12]

<doc> <img>×3 <row> <img>×2 <row> goal: buy the blue shirt in M
choice question: Which element is clicked next? <mask> 1: [1] Add to cart (button) <mask> 2: [2] Size M (radio) <mask> 3: [3] Help (link)
markers [48, 68, 87] · qtype choice · 102 tokens


tensor([[0, 1, 2, 2, 2, 3, 3, 3, 5, 6, 7, 8],
        [0, 1, 2, 3, 4, 2, 3, 4, 5, 6, 7, 8]])

On the page-first layout the image part of the row, ids and positions, is exactly what NeoMME's processor produces for the image alone, and the anchors are the `<mask>` tokens:

In [ ]:
n = len(ref["input_ids"][0])
test_eq(r.ids[:n], ref["input_ids"][0].tolist())
test_eq(r.extras["position_ids"][:, :n], ref["position_ids"][:, 0])
test_eq([r.ids[m] for m in r.markers], [tok.mask_token_id] * 3)
test_eq(r.extras["pixel_values"], ref["pixel_values"])

The `neomme` preset keeps Laya's order, question first, and gives the page its own `<doc>` before the grid, which NeoMME needs to tell the image marker from a patch; a text-only row is `<doc>`, the question and the text, with diagonal positions:

In [ ]:
qfirst = spec.builder(tfms=[Image(side=64)])
rq = qfirst.build_record(rec)[0]
rt = qfirst.build_record({"state": "goal: open the returns page", "questions": qs})[0]
print(show_row(rq, tok)); print(show_row(rt, tok))
test_eq(rq.ids.count(tok.document_token_id), 2)
test_eq(rt.extras["position_ids"], torch.arange(rt.n_tokens).expand(2, -1))
test_eq(tuple(rt.extras["pixel_values"].shape), (0, 3072))

<doc> choice question: Which element is clicked next? <mask> 1: [1] Add to cart (button) <mask> 2: [2] Size M (radio) <mask> 3: [3] Help (link) <doc> <img>×3 <row> <img>×2 <row> goal: buy the blue shirt in M
markers [26, 46, 65] · qtype choice · 103 tokens
<doc> choice question: Which element is clicked next? <mask> 1: [1] Add to cart (button) <mask> 2: [2] Size M (radio) <mask> 3: [3] Help (link)
goal: open the returns page
markers [26, 46, 65] · qtype choice · 93 tokens


Going back, from a row to the picture its pixels show, NeoMME's method of [`pixel_tiles`](01_template.ipynb) puts each patch back on the grid. The grid's height is the number of `<row>` markers in the row, and a patch's numbers are its pixels row by row, three colours each, as `convert_image_to_patches` laid them out. On the fixture screenshot the pixels come back as the screenshot, padded with black to whole patches:

In [ ]:
#| export
def neomme_tiles(processor:NeoMMEProcessor, row:Row, values=None) -> tuple:
    "NeoMME's pixels as the image they show: its patches back on their grid, the normalisation undone; `values`, one per pixel value, summed over the colours"
    pv = torch.as_tensor(row.extras["pixel_values"]).float()
    h = list(row.ids).count(processor.tokenizer.row_token_id)
    if not h or not len(pv): return [], ([] if values is not None else None)
    ip, n, ps = processor.image_processor, len(pv), processor.image_processor.patch_size
    w = n // h
    grid = lambda x: x.reshape(*x.shape[:-2], h, w, ps, ps, 3).swapaxes(-4, -3).reshape(*x.shape[:-2], h * ps, w * ps, 3)
    img = (grid(pv.numpy()) * np.asarray(ip.image_std) + np.asarray(ip.image_mean)).clip(0, 1)
    return [img], (None if values is None else [grid(np.asarray(values)).sum(-1)])

pixel_tiles.register(neomme_tiles)

In [ ]:
shot = np.asarray(PILImage.open("fixtures/shot.png").convert("RGB"), dtype=np.float32) / 255
tiles, vals = pixel_tiles(qfirst.processor, rq, np.ones((2,) + tuple(rq.extras["pixel_values"].shape)))
test_eq((tiles[0].shape, vals[0].shape), ((64, 64, 3), (2, 64, 64)))
test_close(tiles[0][:48], shot, eps=0.01); test_close(tiles[0][48:], 0.0, eps=0.01)      # the screenshot, then black padding to whole patches
test_eq(vals[0][0, 0, 0], 3.0)

A `<doc>` or `<row>` typed into a state is scrubbed like a stray `<mask>`: it would otherwise open a new document or end a patch row.

In [ ]:
rs = qfirst.build_record({"state": "see <doc> and <row> and <mask>", "questions": qs})[0]
test_eq(rs.ids.count(tok.document_token_id), 1)
test_eq(rs.k, 3)

## 4. The spec

`EncoderSpec.from_pretrained` recognises a multimodal checkpoint by the image token in its config and loads its processor; the family entry gives the template (`neomme`) and LoRA's targets (the attention projections `q_proj`, `k_proj`, `v_proj` and `output_projection.o_proj`, leaving the patch embeddings and the MLP alone). NeoMME wants two-axis `position_ids` as (2, batch, length); rows carry them as (2, length), so NeoMME's method of `encoder_inputs` transposes the collated batch, and drops `pixel_values` when no row has an image:

In [ ]:
#| export
def neomme_inputs(config:NeoMMEConfig, inputs:dict) -> dict:
    "NeoMME's inputs: position ids as (2, batch, length), and no `pixel_values` when no row of the batch has an image"
    kw = dict(inputs)
    pos = kw.get("position_ids")
    if pos is not None and pos.dim() == 3 and pos.shape[1] == 2: kw["position_ids"] = pos.transpose(0, 1).contiguous()
    pv = kw.get("pixel_values")
    if pv is not None and pv.shape[0] == 0: kw.pop("pixel_values")
    return kw

encoder_inputs.register(neomme_inputs)

In [ ]:
test_eq(spec.lora_targets, ["q_proj", "k_proj", "v_proj", "o_proj"])
m = set_regime(EncoderDecisionModel(spec.load_encoder(dtype=torch.float32), DecisionHead(spec.width, 0), spec), "lora")
test_eq(sorted({n.split(".lora_")[0].split(".")[-1] for n, p in m.encoder.named_parameters() if p.requires_grad}),
        ["k_proj", "o_proj", "q_proj", "v_proj"])

## 5. One forward pass

A text row and an image row in one batch: one vector per position comes back, the head reads the `<mask>` positions, and each row's logits match its own single-row pass, so padding and mixing change nothing:

In [ ]:
torch.manual_seed(0)
model = EncoderDecisionModel(spec.load_encoder(dtype=torch.float32), DecisionHead(spec.width, layers=0), spec).eval()
b = collate([rq, rt], tok.pad_token_id)
{k: tuple(v.shape) for k, v in b.items()}

{'input_ids': (2, 103),
 'attention_mask': (2, 103),
 'marker_pos': (2, 3),
 'marker_mask': (2, 3),
 'qtype': (2,),
 'target': (2, 3),
 'label': (2,),
 'spans': (2, 3, 2),
 'answerable': (2,),
 'position_ids': (2, 2, 103),
 'pixel_values': (4, 3072)}

In [ ]:
fwd = lambda rows: model(**{k: v for k, v in collate(rows, tok.pad_token_id).items() if k not in ("target", "label")})
with torch.no_grad():
    both, one_img, one_txt = fwd([rq, rt]), fwd([rq]), fwd([rt])
test_close(both[0], one_img[0], eps=1e-4); test_close(both[1], one_txt[0], eps=1e-4)
both.softmax(-1)

tensor([[0.3333, 0.3333, 0.3333],
        [0.3333, 0.3333, 0.3333]])

## 6. A head-only run

The multimodal application runs any encoder that has an adapter, and takes its defaults from it: the row template (its family's preset), the image side (`default_image_side`: 1,024 pixels for NeoMME's pages, one 512-pixel tile for [ModernVBERT](23_modernvbert.ipynb)) and the batching (`prepare_processor`: when the processor tiles, the batches that run the vision tower are divided by the images per row). The caller can override each. The default encoder is ModernVBERT, and section 8 says why. Encoders run in fp32: NeoMME's checkpoint loads in bf16, whose worst-token agreement with fp32 is poor enough to matter for a head trained on frozen features. An `Image` transform resizes each image to `image_side` pixels on its long side, and the frozen encoder fills a cache of anchor vectors, so the images are read once and the export carries no image files. The head is Laya's scorer on the anchors, without Laya's two transformer layers, in every regime, and it standardises NeoMME's vectors before scoring them; both choices come from the real-checkpoint runs of section 7. `head="laya"` asks for the two-layer head. A learner built on data that an earlier learner used gets its own `image_side`: the application replaces the `Image` transform it added before, and keeps one the caller put in the data. Keeping the first learner's side would also find the first learner's cache, and a comparison of image sizes would compare one size with itself.

::: {.callout-note title="Finding: a float16 cache quantises NeoMME's constant channel"}
Channel 139 of NeoMME's output sits at about 30.9 in every vector, with a spread of only 0.02 (the [images tutorial](tutorials/neomme_images.ipynb#why-the-head-standardises-neommes-vectors) measures it). Near 30.9, float16's step is 0.016, so a float16 cache stores that channel as a handful of levels, while the same model run live, as a `Decider` runs it, sees it in float32. A head that standardises its input divides by that small spread, and the two paths then disagree. On the tutorial's digits, the linear head's expected calibration error was 0.106 through `validate` (the cache) and 0.161 through the `Decider` (live), with the same accuracy. With a float32 cache the two agree to the last digit (accuracy 0.41, ECE 0.169). So the multimodal application caches in float32 (`cache_dtype="float32"`). For anchors that costs 4 KB per option, next to the 2 KB of float16. The text application keeps float16.
:::

In [ ]:
#| export
ENCODER = "ModernVBERT/modernvbert"       # the default since 2026-09-30, NeoMME-260M before: section 8 says why

def neomme_image_side(processor:NeoMMEProcessor) -> int:
    "NeoMME's default image side: 1,024 pixels on the long side, for the pages and screenshots it reads"
    return 1024

default_image_side.register(neomme_image_side)

def decision_learner(data:TypedDecisions, encoder=ENCODER, train="head", cache="auto", template=None, image_side="auto",
                     head:str|None=None, dtype=torch.float32, standardize:bool=True, cache_dtype:str="float32", **kw) -> Learner:
    "A multimodal `Learner` for any encoder with an adapter (ModernVBERT by default), with the encoder's template, image side and batching"
    spec = encoder if isinstance(encoder, EncoderSpec) else EncoderSpec.from_pretrained(encoder)
    template = template if template is not None else spec.template.name     # the family's preset, whatever the data was bound to before
    if image_side == "auto": image_side = default_image_side(spec.processor)
    if cache == "auto": cache = "masks" if train == "head" and head in (None, "mlp", "linear") and not kw.get("head_layers") else None
    per_row = prepare_processor(spec.processor, image_side)                 # a tiling processor makes several images of one
    if per_row > 1:           # the vision tower's batch is rows × images per row: fewer rows keep it the size of one image's
        base = PRESETS[kw.get("preset") or default_preset()]
        shrink = ("per_device_eval_batch_size",) if cache else ("per_device_train_batch_size", "per_device_eval_batch_size")
        for k in shrink:      # with cached anchors, only the batches that fill the cache run the vision tower
            if k in base: kw.setdefault(k, max(1, base[k] // per_row))
    if not any(isinstance(t, Image) and not getattr(t, "_added", False) for t in data.tfms):   # the caller's own Image transform is kept
        img = Image(side=image_side); img._added = True       # this call's side, also when an earlier call added one to the same data
        data.tfms = [img] + [t for t in data.tfms if not getattr(t, "_added", False)]
    head = head or "mlp"     # Laya's scorer on the anchors in every regime; Laya's two-layer head trained from scratch collapsed on small image tasks
    return _decision_learner(data, spec, train=train, cache=cache, template=template, head=head, dtype=dtype, standardize=standardize,
                             cache_dtype=cache_dtype, **kw)

Eight browser steps on the fixture screenshot, text rows and image rows mixed; the head trains from the cache and the export loads back and answers for a new screenshot:

In [ ]:
def step(i):
    goal = ["buy the blue shirt", "get help", "add to cart", "open the menu"][i % 4]
    st = {"image": "fixtures/shot.png", "text": f"goal: {goal}"} if i % 2 == 0 else f"goal: {goal}"
    return {"id": f"s{i}", "state": st, "questions": qs, "gold": {"target": {"label": str(1 + i % 3)}}}

mm = TypedDecisions.from_records([step(i) for i in range(8)], valid=0.25, calib=0)
learn = decision_learner(mm, FIX, image_side=64, preset="cpu", loss="soft_ce", cache_dir=tempfile.mkdtemp())
learn.fit(3, lr=1e-2)
out = learn.export(Path(tempfile.mkdtemp()) / "browser-jev")
jev = Decider.load(out, device="cpu")
jev.predict({"image": "fixtures/shot.png", "text": "goal: open the returns page"}, qs)

{'eval_loss': '1.024', 'eval_accuracy': '0.5', 'eval_soft_accuracy': '0.3674', 'eval_brier': '0.6133', 'eval_kl': '1.024', 'eval_tv': '0.6326', 'eval_ece': '0.06849', 'eval_score_mae': 'nan', 'eval_within_one': 'nan', 'eval_n': 2, 'eval_accuracy_choice': '0.5', 'eval_accuracy_score': 'nan', 'eval_accuracy_noul': 'nan', 'eval_n_unanswerable': 0, 'eval_runtime': '0.0015', 'eval_samples_per_second': '1366', 'eval_steps_per_second': '683', 'epoch': '1'}
{'eval_loss': '1.262', 'eval_accuracy': '0', 'eval_soft_accuracy': '0.2859', 'eval_brier': '0.7801', 'eval_kl': '1.262', 'eval_tv': '0.7141', 'eval_ece': '0.4355', 'eval_score_mae': 'nan', 'eval_within_one': 'nan', 'eval_n': 2, 'eval_accuracy_choice': '0', 'eval_accuracy_score': 'nan', 'eval_accuracy_noul': 'nan', 'eval_n_unanswerable': 0, 'eval_runtime': '0.0016', 'eval_samples_per_second': '1243', 'eval_steps_per_second': '621.6', 'epoch': '2'}
{'eval_loss': '1.295', 'eval_accuracy': '0', 'eval_soft_accuracy': '0.2777', 'eval_brier': '0.7

{'target': {'type': 'choice',
  'choice': '2',
  'probabilities': {'1': 0.3033, '2': 0.3601, '3': 0.3366},
  'confidence': 0.0022,
  'answer_confidence': 0.3601}}

In [ ]:
test_eq(sorted(p.name for p in out.iterdir()), ["encoder", "head.safetensors", "processor", "sysone.json"])
test_eq(jev.predict(step(0)["state"], qs), learn.predict(step(0)["state"], qs))
test_eq("anchors" in learn.dataset("train").column_names, True)
test_eq((default_image_side(proc), prepare_processor(proc, 64), learn.data.builder.template.name), (1024, 1, "neomme"))   # NeoMME's defaults
test_eq(learn.dataset("train").features["anchors"].dtype, "float32")      # the application's cache keeps float32
sides = lambda l: [t.side for t in l.data.builder.tfms if isinstance(t, Image)]
test_eq(sides(decision_learner(mm, FIX, image_side=32, preset="cpu", loss="soft_ce", cache_dir=tempfile.mkdtemp())), [32])
own = TypedDecisions.from_records([step(i) for i in range(8)], valid=0.25, calib=0, tfms=[Image(side=48)])
test_eq(sides(decision_learner(own, FIX, preset="cpu", loss="soft_ce", cache_dir=tempfile.mkdtemp())), [48])
fresh = subprocess.run([sys.executable, "-c", "import sysone.multimodal, sysone.learner as L; print(hasattr(L.Learner, 'fit_linear'))"],
                       capture_output=True, text=True)
test_eq(fresh.stdout.strip(), "True")      # in a fresh session the application brings the cache's verbs, fit_linear among them

In [ ]:
test_eq(list(learn.dataset("train")["case"]), learn.data.rows("train")["case"])     # image rows keep their case ids in the cache

The feature cache's key covers the images, not only the tokens: the same steps over a different screenshot of the same size get a cache of their own.

In [ ]:
from sysone.cache import FeatureCache

In [ ]:
other = Path(tempfile.mkdtemp()) / "other.png"
PILImage.open("fixtures/shot.png").transpose(PILImage.Transpose.FLIP_LEFT_RIGHT).save(other)
def steps_on(img): return TypedDecisions.from_records([{**step(i), "state": {**step(i)["state"], "image": str(img)}} if i % 2 == 0 else step(i)
                                                        for i in range(8)], calib=0, tfms=[Image(side=64)]).bind(proc, "neomme")
ka = FeatureCache(learn.model, steps_on("fixtures/shot.png"), "masks").key("train")
kb = FeatureCache(learn.model, steps_on(other), "masks").key("train")
test_ne(ka, kb)

With `train="lora"` the encoder trains through adapters and the cache is dropped; text-only records work in the same call, with no image key. On the real checkpoint:

In [ ]:
#| eval: false
data  = TypedDecisions.from_json("steps.jsonl", valid=0.1)        # or TypedDecisions.from_registry("mind2web_mm")
learn = decision_learner(data, "Hcompany/NeoMME-260M", train="head", image_side=1024)
learn.show_batch(2)
learn.lr_find()
learn.fit_one_cycle(3, lr=1e-4)
learn.calibrate()
learn.export("browser-jev")

## 7. Two small image tasks on the real checkpoint

The fixture shows that the pieces fit; only the real checkpoint can show that what NeoMME reads off an image reaches the anchors. Two small tasks, the first a **choice** and the second a **noul**:

- *Which digit is written in the image?*, ten options: [MNIST](https://huggingface.co/datasets/ylecun/mnist), 10 images per digit to train (a fifth of the cases held out for the temperatures) and 10 others per digit, from MNIST's test split, to score; chance is 0.10;
- *The photo shows a hot dog.*: 40 photos of each class to train and 40 others of each class to score, from [truepositive/hotdog_nothotdog](https://huggingface.co/datasets/truepositive/hotdog_nothotdog), stock photos of hot dogs and of other food, 224 px on their long side; it stands in for Kaggle's `dansbecker/hot-dog-not-hot-dog`, which needs a Kaggle account to download: with one, `kaggle datasets download dansbecker/hot-dog-not-hot-dog` and the same function over its folders. Chance is 0.50.

A state is JSON, so an image travels as a path; the `Image` transform the application adds loads it and resizes it, which here enlarges both: the 28 px digits to 224 px on the long side, the 224 px photos to 384 px. The [images tutorial](tutorials/neomme_images.ipynb) runs these tasks with its outputs: the rows, the training curves, the answers image by image and the confusions.

In [ ]:
#| eval: false
import random
from datasets import load_dataset
from huggingface_hub import snapshot_download
out = Path("neomme-real")

def mnist_records(per_class=10, seed=0):
    "Choice records over MNIST: per_class images of each digit from train, and as many from test"
    rng = random.Random(seed)
    ds, q, recs = load_dataset("ylecun/mnist"), {"digit": {"type": "choice", "instructions": "Which digit is written in the image?",
                                                           "criteria": [str(d) for d in range(10)]}}, {}
    for split in ("train", "test"):
        by = {d: [] for d in range(10)}
        for i in rng.sample(range(len(ds[split])), len(ds[split])):
            if len(by[ds[split][i]["label"]]) < per_class: by[ds[split][i]["label"]].append(i)
            if all(len(v) == per_class for v in by.values()): break
        (out / "images" / split).mkdir(parents=True, exist_ok=True); recs[split] = []
        for d, idx in by.items():
            for i in idx:
                p = out / "images" / split / f"{i}.png"; ds[split][i]["image"].save(p)
                recs[split].append({"id": f"mnist-{split}-{i}", "state": {"text": "", "image": str(p)}, "questions": q, "gold": {"digit": {"label": str(d)}}})
        rng.shuffle(recs[split])
    return recs["train"], recs["test"]

def hotdog_records(per_class=40, root=None, seed=0):
    "Noul records over hot dog / not hot dog photos: per_class of each class to train, per_class others to score"
    rng = random.Random(seed)
    root = Path(root or snapshot_download("truepositive/hotdog_nothotdog", repo_type="dataset")) / "train"
    q, train, test = {"hot_dog": {"type": "noul", "instructions": "The photo shows a hot dog."}}, [], []
    for cls, truth in (("hotdog", 1.0), ("not_hotdog", 0.0)):
        files = sorted((root / cls).glob("*.jpg")); rng.shuffle(files)
        for part, fs in ((train, files[:per_class]), (test, files[per_class:2 * per_class])):
            part += [{"id": f"{cls}-{p.stem}", "state": {"text": "", "image": str(p)}, "questions": q, "gold": {"hot_dog": {"noul": truth}}} for p in fs]
    rng.shuffle(train); rng.shuffle(test)
    return train, test

The application's default path, the encoder frozen and its vectors at the anchors cached, with the head it trains by default (Laya's scorer) and with the linear head fitted by LBFGS; then LoRA:

In [ ]:
#| eval: false
train, test = mnist_records()                       # or hotdog_records(), with image_side=384
for head in ("mlp", "linear"):
    data = TypedDecisions.from_records(train, valid=test, calib=0.2)
    learn = decision_learner(data, "Hcompany/NeoMME-260M", train="head", head=head, image_side=224)
    if head == "linear": learn.fit_linear()
    else: learn.fit_one_cycle(30, lr=1e-3)
    print(head, learn.validate()["accuracy"])

learn = decision_learner(data, "Hcompany/NeoMME-260M", train="lora", image_side=224)    # the encoder trains through adapters
learn.fit_one_cycle(20, lr=1e-3)
learn.validate()["accuracy"]

Run on 2026-09-26, M3 Pro (MPS, fp32). Accuracy on the held-out images (100 digits, 80 photos), before and after the two fixes described below:

| | digits, 10-way choice | hot dog, noul |
| --- | --- | --- |
| chance | 0.10 | 0.50 |
| a linear probe on the image alone (the mean of its patch vectors) | 0.56 | 0.71 |
| *before* · head only, Laya's scorer, `lr_find`'s valley, 10 epochs | 0.16 | 0.50 |
| *before* · head only, the linear head (`fit_linear`) | 0.40 | 0.70 |
| *before* · LoRA, Laya's two-layer head, `lr_find`'s valley, 10 epochs | 0.10 | 0.50 |
| *after* · head only, Laya's scorer, `lr_find`'s valley, 10 epochs | 0.21 | 0.70 |
| *after* · head only, Laya's scorer, 10⁻³, 30 epochs | 0.29 | 0.76 |
| *after* · head only, the linear head | 0.40 | 0.70 |
| *after* · LoRA, Laya's two-layer head, 10⁻³, 20 epochs | 0.12 | 0.53 |
| *after* · LoRA, Laya's scorer alone (the default now), 10⁻³, 20 epochs | 0.34 | 0.74 |

Both kinds of question work on images: from 80 to 100 examples the best configurations read the digits at four times chance and the photos at three in four. The probe reads NeoMME's image representation without the question, the options or the head, and says the images reach the model and carry the class. For scale, a logistic regression on the digits' raw 28 × 28 pixels, on the same 100 images, scores 0.77: NeoMME is a page encoder, and a digit at 224 px is a 7 × 7 grid of its 32-pixel patches.

::: {.callout-note title="Finding: NeoMME's vectors need standardising, and a small task needs no head layers"}
Before the fixes, the default head barely learned (0.16 on the digits, 0.50 on the photos) and LoRA collapsed to a uniform answer (every option at the same probability), while the linear head, which standardises its inputs, reached 0.40 and 0.70. One channel of NeoMME's vectors carries about 97% of every vector's norm. A LayerNorm scales each vector by its own spread, so after it the dominant channel still dwarfs the others and every anchor looks alike; the linear head's per-dimension standardisation puts every channel on the same scale. In a plain Adam loop over the cached anchors, Laya's scorer went from 0.18 on the raw digits' anchors to 0.40 on standardised ones, and from 0.73 to 0.78 on the photos. So the multimodal application's heads now standardise the encoder's output per dimension (`DecisionHead(standardize=True)`, fitted from the training rows before the first fit; see the [models](03_models.ipynb#standardising-the-encoders-output) notebook).

That fixed the default head on the photos (0.50 to 0.70) but not LoRA with Laya's head, whose two transformer layers, trained from scratch on 80 examples, still collapsed at every learning rate tried; with Laya's scorer alone LoRA learns (0.34 and 0.74), so that is now the application's head in every regime. The images reach the model on every path: zeroing a batch's pixels changes its logits. On so few examples `lr_find`'s valley is too cautious (1.3e-04 for the head here, 1.4e-06 for LoRA on the photos): the tables' best runs use 10⁻³.
:::

The [images tutorial](tutorials/neomme_images.ipynb) reruns both tasks with their outputs. It caches the anchors in float32, as the application now does (see the finding in section 6), which moves the heads' results by a point or two: on the digits the linear head scores 0.41 and the default head 0.31. It also adds two things. Both come from single runs scored on the test split, so they are leads rather than settings:

- **Smaller images read better.** With the linear head the digits reach 0.57 at 112 px (4 × 4 patches) and 0.54 at 56 px, against 0.41 at the 224 px used above, and the default head reaches 0.57 at 56 px. The photos lose accuracy below their own 224 px and gain nothing clear from being enlarged past it.
- **The photo split is kinder than it looks.** Stock photos come in shoots, several shots of one scene, and a seeded split puts 17 of the 40 held-out hot dogs next to a training photo from the same shoot. The linear head finds 13 of those 17 hot dogs, but only 13 of the 23 without one. A split by shoot would be the honest one, as sysone splits by case (see [data](02_data.ipynb#splits-by-case)).

## 8. Which encoder by default

The application's default was NeoMME-260M, because H company built it for screenshots and documents, the pages a browser agent decides over. Three comparisons have since measured it against ModernVBERT, each with the same records and the same recipe for both, and each is a tutorial with every output:

| | NeoMME-260M | ModernVBERT |
| --- | --- | --- |
| 100 MNIST digits, a 10-way choice ([comparison tutorial](tutorials/neomme_vs_modernvbert.ipynb)) | 0.31 | 0.84 |
| 80 photos, a hot-dog noul (the same tutorial) | 0.74 | 0.94 |
| Web pages as text: the 12-way element choice on the test and `ood` websites ([text tutorial](tutorials/web_text.ipynb)) | 0.406 and 0.446 | 0.384 and 0.446 |
| Web pages with the screenshot, a pilot: the operation question ([screens tutorial](tutorials/web_screens.ipynb)) | 0.812 | 0.885 |
| The same pilot: the element to act on | 0.136 | 0.209 |
| Seconds to encode a screenshot row at 1,024 pixels | 0.58 | 0.61 |

ModernVBERT wins every comparison that uses an image, including real screenshots, the one where NeoMME's design should have favoured it. On text the two are level. So since 2026-09-30 the default is ModernVBERT. The screenshot comparison is a pilot, 276 training steps in which no head learned to pick the element to act on, and the switch didn't wait for a full run, which could still revise it. NeoMME-260M is one argument away: `decision_learner(data, "Hcompany/NeoMME-260M")`. For screenshots, give ModernVBERT more than its default single tile: the screens tutorial reads them at `image_side=1024`, 2 × 2 tiles and the global view.

## 9. What did not work, and what to watch

- **The processor's own rows.** `NeoMMEProcessor` rejects text and an image in one row, in either order, and two images in one row; rows are built here instead, with the processor's image processor for the patches.
- **Spaces after `<mask>`.** `"<mask> Save"` tokenizes as `<mask> ĠS ave`, fragmenting the option's first word; `"<mask>Save"` keeps it whole. The presets write no space.
- **A second `<doc>`.** In the question-first layout the image needs its own `<doc>`: without it the model counts the `<img>` marker as a patch and the forward fails on the patch count.
- **Missing positions.** Leaving `position_ids` out of an image row raises nothing and silently falls back to one-axis positions; every row therefore carries them.
- **No resolution cap.** The base checkpoints' processor has no `max_side`, so a 4K screenshot becomes 11,750 patches; the `Image` transform resizes first (1,024 px on the long side: 32 × 24 patches for a 1,024 × 768 shot).
- **bf16.** The checkpoint's default dtype; fp32 is the application's default, bf16 and fp16 both run on MPS.
- **Raw anchor geometry.** NeoMME's final norm has no scale, one channel carries about 97% of every vector's norm, and raw cosines between anchors are all about 0.99. A raw cosine cannot rank options, and neither can a head trained on the raw vectors: the scorer's LayerNorm does not undo it (section 7), so the application standardises the head's input per dimension.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()